# Step 1: Install Required Libraries

In this step, we install the required libraries for running BERT-based sentiment analysis with Hugging Face Transformers and PyTorch.

In [ ]:
!pip install -q transformers torch accelerate

# Step 2: Import Libraries

We import the Hugging Face pipeline API to perform sentiment analysis inference with a pretrained transformer model.

In [ ]:
from transformers import pipeline

# Step 3: Load a Pretrained Sentiment Analysis Pipeline

We load a pretrained sentiment analysis model using the Hugging Face pipeline API.

In [ ]:
classifier = pipeline("sentiment-analysis")

# Step 4: Test the Model on Example Reviews

We test the pretrained BERT-based sentiment classifier on a few movie review examples.

In [ ]:
reviews = [
    "This movie was absolutely amazing. I loved every minute of it.",
    "This was one of the worst films I have ever seen.",
    "The acting was decent, but the story felt weak and predictable.",
    "A brilliant film with powerful performances and excellent direction."
]

for review in reviews:
    result = classifier(review)
    print("Review:", review)
    print("Prediction:", result[0]["label"])
    print("Score:", result[0]["score"])
    print("-" * 80)

# Step 5: Create a Simple Custom Prediction Function

We create a helper function so that we can easily test any custom movie review.

In [ ]:
def predict_review_sentiment(text):
    result = classifier(text)[0]
    return {
        "review": text,
        "prediction": result["label"],
        "score": round(result["score"] * 100, 2)
    }

# Step 6: Test a Custom Review

We test the function on a single custom review input.

In [ ]:
custom_review = "This movie had an emotional story and fantastic acting."
prediction = predict_review_sentiment(custom_review)
prediction

# Step 7: Save Results for Documentation

We save a few sample predictions that can later be added to the README file or GitHub project description.

In [ ]:
sample_reviews = [
    "This movie was absolutely amazing.",
    "I did not enjoy this movie at all.",
    "The visuals were beautiful, but the plot was weak."
]

for review in sample_reviews:
    print(predict_review_sentiment(review))

# Step 8: Install Additional Libraries for Fine-Tuning

We install the datasets and evaluate libraries required for dataset handling and model evaluation.

In [ ]:
!pip install -q datasets evaluate

# Step 9: Import Training Libraries

We import the tokenizer, model, dataset utilities, and Trainer components required for BERT fine-tuning.

In [ ]:
import numpy as np
from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# Step 10: Load the IMDB Dataset

We load the IMDB dataset for binary sentiment classification.

In [ ]:
dataset = load_dataset("imdb")
dataset

# Step 11: Load the Tokenizer

We use a pretrained DistilBERT tokenizer for text preprocessing.

In [ ]:
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Step 12: Tokenize the Dataset

We tokenize the text data so it can be used by the transformer model.

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Step 13: Prepare a Smaller Training Subset

To keep training faster in Colab, we create smaller train and test subsets.

In [ ]:
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

# Step 14: Load the Sequence Classification Model

We load a pretrained transformer model for binary sentiment classification.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Step 15: Prepare Evaluation Metrics

We use accuracy as the main evaluation metric for this project.

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

# Step 16: Create the Data Collator

The data collator dynamically pads input sequences during training.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Step 17: Define Training Arguments

We define the main training configuration for fine-tuning the transformer model.

In [ ]:
training_args = TrainingArguments(
    output_dir="imdb_bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="logs",
    report_to="none"
)

# Step 18: Initialize the Trainer

We create a Trainer instance to manage the training and evaluation workflow.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Step 19: Fine-Tune the Model

We fine-tune the pretrained DistilBERT model on the IMDB sentiment dataset.

In [ ]:
trainer.train()

# Step 20: Evaluate the Fine-Tuned Model

We evaluate the model on the test subset.

In [ ]:
trainer.evaluate()

# Step 21: Save the Fine-Tuned Model

We save the fine-tuned transformer model and tokenizer for later inference or deployment.

In [ ]:
model.save_pretrained("imdb_distilbert_model")
tokenizer.save_pretrained("imdb_distilbert_model")

# Step 22: Run Inference with the Fine-Tuned Model

We create a Hugging Face pipeline using the fine-tuned model.

In [ ]:
from transformers import pipeline

fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model="imdb_distilbert_model",
    tokenizer="imdb_distilbert_model"
)

# Step 23: Test the Fine-Tuned Model

We test the saved model on new review examples.

In [ ]:
label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Positive"
}

for review in test_reviews:
    result = fine_tuned_classifier(review)
    raw_label = result[0]["label"]
    mapped_label = label_map.get(raw_label, raw_label)

    print("Review:", review)
    print("Prediction:", mapped_label)
    print("Score:", result[0]["score"])
    print("-" * 80)